# Random Forest Classification Workflow

This notebook loads the yearly training/validation/test CSV files, adds NDVI and NDSI, removes missing rows, standardizes predictors, tunes a Random Forest with `GridSearchCV`, and evaluates performance on validation and test sets.


In [ ]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

pd.set_option("display.max_columns", None)


## 1. Load data

If your files are in a different folder, change `DATA_DIR` below.


In [ ]:
# Import code for reading in github data
from google.colab import drive
drive.mount('/content/drive')

import os

base_path = "/content/drive/MyDrive/Python_Project"

import pandas as pd

train_2020 = pd.read_csv(os.path.join(base_path, 'Train_2020.csv'))
train_2021 = pd.read_csv(os.path.join(base_path, 'Train_2021.csv'))
train_2022 = pd.read_csv(os.path.join(base_path, 'Train_2022.csv'))
train_2023 = pd.read_csv(os.path.join(base_path, 'Train_2023.csv'))

val_2024 = pd.read_csv(os.path.join(base_path, 'Val_2024.csv'))
test_2025 = pd.read_csv(os.path.join(base_path, 'Test_2025.csv'))

train_2020.head()

Mounted at /content/drive


,Class,X,Y,Blue,Green,Red,NIR,SWIR
0,Cloud,591159.4361,5.220529e+06,36181.0,38428.0,41335.0,43454.0,34017.0
1,Cloud,590806.7570,5.220482e+06,34072.0,35740.0,37728.0,38914.0,26475.0
2,Cloud,627264.7196,5.191233e+06,33891.0,37152.0,39247.0,42225.0,28764.0
3,Cloud,590619.3083,5.220389e+06,42154.0,46492.0,65535.0,52709.0,39955.0
4,Cloud,590642.7787,5.220514e+06,36983.0,39379.0,41847.0,43215.0,32189.0


## 2. Add NDVI and NDSI

Formulas used:
- `NDVI = (NIR - Red) / (NIR + Red)`
- `NDSI = (Green - SWIR) / (Green + SWIR)`


In [ ]:
def add_indices(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["NDVI"] = (df["NIR"] - df["Red"]) / (df["NIR"] + df["Red"] + 1e-10)
    df["NDSI"] = (df["Green"] - df["SWIR"]) / (df["Green"] + df["SWIR"] + 1e-10)
    return df

train_2020 = add_indices(train_2020)
train_2021 = add_indices(train_2021)
train_2022 = add_indices(train_2022)
train_2023 = add_indices(train_2023)
val_2024 = add_indices(val_2024)
test_2025 = add_indices(test_2025)

train_2020[["NDVI", "NDSI"]].head()


,NDVI,NDSI
0,0.024991,0.060888
1,0.015475,0.148919
2,0.036552,0.127253
3,-0.108471,0.075619
4,0.016082,0.100464


## 3. Combine training years and inspect missing values


In [ ]:
train_df = pd.concat(
    [train_2020, train_2021, train_2022, train_2023],
    ignore_index=True
)

print("Combined training shape:", train_df.shape)
print("\nTraining null counts:")
print(train_df.isnull().sum())
print("\nValidation null counts:")
print(val_2024.isnull().sum())
print("\nTest null counts:")
print(test_2025.isnull().sum())


Combined training shape: (48000, 10)

Training null counts:
Class      0
X          0
Y          0
Blue     217
Green    215
Red      217
NIR      215
SWIR     215
NDVI     217
NDSI     215
dtype: int64

Validation null counts:
Class    0
X        0
Y        0
Blue     0
Green    0
Red      0
NIR      0
SWIR     0
NDVI     0
NDSI     0
dtype: int64

Test null counts:
Class    0
X        0
Y        0
Blue     0
Green    0
Red      1
NIR      0
SWIR     0
NDVI     1
NDSI     0
dtype: int64


In [ ]:
train_df = train_df.dropna().copy()
val_2024 = val_2024.dropna().copy()
test_2025 = test_2025.dropna().copy()

print("Training shape after dropna:", train_df.shape)
print("Validation shape after dropna:", val_2024.shape)
print("Test shape after dropna:", test_2025.shape)


Training shape after dropna: (47783, 10)
Validation shape after dropna: (12000, 10)
Test shape after dropna: (11999, 10)


## 4. Define predictors and target

Including:
- spectral bands
- NDVI and NDSI
- X and Y coordinates


In [ ]:
target = "Class"
features = ["Blue", "Green", "Red", "NIR", "SWIR", "NDVI", "NDSI", "X", "Y"]

X_train = train_df[features]
y_train = train_df[target]

X_val = val_2024[features]
y_val = val_2024[target]

X_test = test_2025[features]
y_test = test_2025[target]

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)
print("\nClass counts in training data:")
print(y_train.value_counts())


X_train shape: (47783, 9)
X_val shape: (12000, 9)
X_test shape: (11999, 9)

Class counts in training data:
Class
Cloud    16000
Other    15998
Snow     15785
Name: count, dtype: int64


## 5. Standardize predictors


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Scaled training array shape:", X_train_scaled.shape)


Scaled training array shape: (47783, 9)


## 6. Baseline Random Forest


In [ ]:
baseline_rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

baseline_rf.fit(X_train_scaled, y_train)

val_pred_baseline = baseline_rf.predict(X_val_scaled)

print("Baseline validation accuracy:", accuracy_score(y_val, val_pred_baseline))
print("\nBaseline validation classification report:")
print(classification_report(y_val, val_pred_baseline))


Baseline validation accuracy: 0.9945833333333334

Baseline validation classification report:
              precision    recall  f1-score   support

       Cloud       0.99      0.99      0.99      4000
       Other       1.00      0.99      1.00      4000
        Snow       0.99      1.00      0.99      4000

    accuracy                           0.99     12000
   macro avg       0.99      0.99      0.99     12000
weighted avg       0.99      0.99      0.99     12000



## 7. Tune model with GridSearchCV

This tunes the Random Forest on the training data using cross-validation.


In [ ]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", "log2"]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid=param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_scaled, y_train)

print("Best parameters:")
print(grid_search.best_params_)
print("\nBest cross-validation score:")
print(grid_search.best_score_)


Fitting 3 folds for each of 48 candidates, totalling 144 fits
Best parameters:
{'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}

Best cross-validation score:
0.986501273057824


## 8. Evaluate best model on validation data


In [ ]:
best_rf = grid_search.best_estimator_

val_pred = best_rf.predict(X_val_scaled)

print("Tuned validation accuracy:", accuracy_score(y_val, val_pred))
print("\nTuned validation classification report:")
print(classification_report(y_val, val_pred))
print("\nValidation confusion matrix:")
print(confusion_matrix(y_val, val_pred))


Tuned validation accuracy: 0.99525

Tuned validation classification report:
              precision    recall  f1-score   support

       Cloud       0.99      0.99      0.99      4000
       Other       1.00      0.99      1.00      4000
        Snow       0.99      1.00      1.00      4000

    accuracy                           1.00     12000
   macro avg       1.00      1.00      1.00     12000
weighted avg       1.00      1.00      1.00     12000


Validation confusion matrix:
[[3968    5   27]
 [  20 3980    0]
 [   2    3 3995]]


## 9. Feature importance


In [ ]:
feature_importance = pd.DataFrame({
    "Feature": features,
    "Importance": best_rf.feature_importances_
}).sort_values("Importance", ascending=False)

feature_importance


,Feature,Importance
6,NDSI,0.275658
0,Blue,0.251017
1,Green,0.209530
2,Red,0.097154
4,SWIR,0.075451
3,NIR,0.044003
5,NDVI,0.021214
7,X,0.017803
8,Y,0.008169


## 10. Save validation model



In [ ]:
val_results = val_2024.copy()
val_results["Predicted_Class"] = val_pred

output_path_val = os.path.join(base_path, "rf_val_predictions.csv")
val_results.to_csv(output_path_val, index=False)

print("Saved validation predictions to:", output_path_val)

Saved validation predictions to: /content/drive/MyDrive/Python_Project/rf_val_predictions.csv


## 11. Optional final model using train + validation

After choosing the best hyperparameters based on validation performance, you can retrain using both the original training data and validation data, then evaluate once on the held-out test set.


In [ ]:
train_val_df = pd.concat([train_df, val_2024], ignore_index=True)

X_train_val = train_val_df[features]
y_train_val = train_val_df[target]

final_scaler = StandardScaler()
X_train_val_scaled = final_scaler.fit_transform(X_train_val)
X_test_final_scaled = final_scaler.transform(X_test)

final_rf = RandomForestClassifier(
    **grid_search.best_params_,
    random_state=42,
    n_jobs=-1
)

final_rf.fit(X_train_val_scaled, y_train_val)

test_pred = final_rf.predict(X_test_final_scaled)

print("Final test accuracy:", accuracy_score(y_test, test_pred))
print("\nFinal test classification report:")
print(classification_report(y_test, test_pred))
print("\nFinal test confusion matrix:")
print(confusion_matrix(y_test, test_pred))


Final test accuracy: 0.7100591715976331

Final test classification report:
              precision    recall  f1-score   support

       Cloud       0.58      0.98      0.73      4000
       Other       0.86      0.98      0.91      4000
        Snow       1.00      0.17      0.29      3999

    accuracy                           0.71     11999
   macro avg       0.81      0.71      0.64     11999
weighted avg       0.81      0.71      0.64     11999


Final test confusion matrix:
[[3921   79    0]
 [  78 3922    0]
 [2738  584  677]]


## 12. Save predictions (optional)


In [ ]:
test_results = test_2025.copy()
test_results["Predicted_Class"] = test_pred

output_path = os.path.join(base_path, "rf_test_predictions.csv")
test_results.to_csv(output_path, index=False)

print("Saved predictions to:", output_path)


Saved predictions to: /content/drive/MyDrive/Python_Project/rf_test_predictions.csv
